In [ ]:
from sickle import Sickle
import pandas as pd
import time

sickle = Sickle('https://tailieuso.tlu.edu.vn/oai/request',verify=False)


records = sickle.ListRecords(metadataPrefix='oai_dc', ignore_deleted=True)

data = []
count = 0
max_records = 5000  

print("Đang thu thập dữ liệu...")

try:
    for record in records:
        # Lấy thông tin trong thẻ <metadata>
        if 'metadata' in record.raw:
            meta = record.metadata
            
            # Trích xuất các trường quan trọng
            title = meta.get('title', [''])[0]
            creators = '; '.join(meta.get('creator', []))
            subjects = '; '.join(meta.get('subject', [])) # Từ khóa
            description = '; '.join(meta.get('description', [])) # Tóm tắt/Abstract
            date = meta.get('date', [''])[0]
            publisher = meta.get('publisher', [''])[0]
            language = meta.get('language', [''])[0]
            identifier = meta.get('identifier', [''])[0] # Link gốc
            doc_type_list = meta.get('type', []) 
            doc_type = '; '.join(doc_type_list)
            data.append({
                'id': record.header.identifier,
                'title': title,
                'creators': creators,
                'subjects': subjects,
                'abstract': description, 
                'date': date,
                'document_type': doc_type,
                'publisher': publisher,
                'language':     language,
                'link': identifier
            })
            
            count += 1
            if count % 100 == 0:
                print(f"Đã lấy {count} bản ghi...")
            
            if count >= max_records:
                break
                
except Exception as e:
    print(f"Lỗi hoặc hết dữ liệu: {e}")


df = pd.DataFrame(data)
df.to_csv('tlu_metadata.csv', index=False, encoding='utf-8-sig')
print(f"Hoàn thành! Đã lưu {len(df)} dòng vào file tlu_metadata.csv")


In [ ]:
from sickle import Sickle

sickle = Sickle('https://tailieuso.tlu.edu.vn/oai/request', verify=False)

print("=== CÁC METADATA FORMAT HỖ TRỢ ===\n")

formats = sickle.ListMetadataFormats()

for fmt in formats:
    print(f"Prefix: {fmt.metadataPrefix}")
    print(f"  Schema: {fmt.schema}")
    print(f"  Namespace: {fmt.metadataNamespace}")
    print("-" * 70)

In [ ]:
import pandas as pd
df = pd.read_csv('tlu_metadata.csv')
df.head()


In [ ]:
df.columns

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df.document_type.value_counts()

In [28]:
df[df.document_type == 'BB; LT']

,title,creators,subjects,abstract,date,document_type,link
4281,Local Trichoderma strains as a control strateg...,"Abd-El-Kareem, Farid",Biocontrol; Black root rot; Enzyme activity; S...,Each of the antagonistic fungal strains signif...,2021-08-17T08:17:25Z,BB; LT,2522-8307


In [ ]:
import pandas as pd
df = pd.read_json('tlu_metadata_clean.jsonl', lines=True)

print(f"Tổng số document: {len(df)}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated(subset=['id']).sum()}")
print(f"\nSample authors format:\n{df['authors'].head(3)}")


Tổng số document: 5000

Null values:
id              0
uri             0
oai_id          0
type            0
raw_type        0
title           0
abstract        0
language        0
publisher       0
authors         0
advisors        0
subjects        0
year            0
date            0
degree       3247
major           0
ddc             0
dtype: int64

Duplicates: 0

Sample authors format:
0    [Đinh Văn Thạch]
1    [Lưu Mạnh Quảng]
2       [Mẫn Văn Thể]
Name: authors, dtype: object
